In [1]:
import numpy as np

In [51]:
X_orig = np.array(([50] * 10 + [5] * 20))
print(X_orig)

[50 50 50 50 50 50 50 50 50 50  5  5  5  5  5  5  5  5  5  5  5  5  5  5
  5  5  5  5  5  5]


In [52]:
X_orig_mean = np.mean(X_orig)
X_orig_var = np.var(X_orig)
print(X_orig_mean, X_orig_var)

20.0 450.0


In [53]:
N = len(X_orig)

K = 10_000  # number of estimates
n = 5  # mini-batch size

In [55]:
rng = np.random.default_rng(0)
# K mini-batches, each of size n, sampled uniformly with replacement
samples_A = rng.choice(X_orig, size=(K, n), replace=True)
samples_A

array([[ 5,  5,  5, 50, 50],
       [50, 50, 50, 50,  5],
       [ 5,  5,  5,  5,  5],
       ...,
       [ 5,  5,  5, 50,  5],
       [50,  5,  5,  5,  5],
       [50,  5, 50,  5,  5]], shape=(10000, 5))

In [56]:
# one estimate per row → K estimates
estimates_A = samples_A.mean(axis=1)
estimates_A

array([23., 41.,  5., ..., 14., 14., 23.], shape=(10000,))

In [57]:
print("Mean of estimates :", estimates_A.mean())  # ≈ 20
print("Var  of estimates :", estimates_A.var())  # ≈ 450 / n = 90
true_mean = X_orig.mean()  # 20.0
bias_A = estimates_A.mean() - true_mean
print("Bias :", bias_A)

Mean of estimates : 19.9553
Var  of estimates : 89.94670190999999
Bias : -0.04469999999999885


In [58]:
# Experiment C
# Proposal distribution q: favor 50s
# Each 50 gets 0.07, each 5 gets 0.015 → totals: 10*0.07 + 20*0.015 = 0.7 + 0.3 = 1.0 ✓
q = np.array([0.07] * 10 + [0.015] * 20)
# Sanity check
assert np.isclose(q.sum(), 1.0)
q

array([0.07 , 0.07 , 0.07 , 0.07 , 0.07 , 0.07 , 0.07 , 0.07 , 0.07 ,
       0.07 , 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015,
       0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015,
       0.015, 0.015, 0.015])

In [59]:
# Sample K mini-batches of size n from q (non-uniform)
idx = rng.choice(N, size=(K, n), replace=True, p=q)
idx

array([[ 6, 22,  8,  3, 18],
       [14,  4,  6,  1,  7],
       [15,  1,  3, 13,  4],
       ...,
       [ 4,  5,  9,  6,  1],
       [ 9,  3,  3, 19,  6],
       [ 9,  7,  8, 16,  2]], shape=(10000, 5))

In [60]:
samples_B = X_orig[idx]

In [61]:
samples_B

array([[50,  5, 50, 50,  5],
       [ 5, 50, 50, 50, 50],
       [ 5, 50, 50,  5, 50],
       ...,
       [50, 50, 50, 50, 50],
       [50, 50, 50,  5, 50],
       [50, 50, 50,  5, 50]], shape=(10000, 5))

In [ ]:
# Naive estimator: just average (no correction)
estimates_B_naive = samples_B.mean(axis=1)
print("Mean of estimates :", estimates_B_naive.mean())  # expect > 20 (biased high)
print(
    "Var  of estimates :", estimates_B_naive.var()
)  # expect < 90 (lower variance)         # 20.0
bias_B = estimates_B_naive.mean() - true_mean
print("Bias in B :", bias_B)

Mean of estimates : 36.5108
Var  of estimates : 85.87608336
Bias in B : 16.510800000000003


In [ ]:
# Importance weights: same shape as X_orig, parallel array
w = (1 / N) / q
w

array([0.47619048, 0.47619048, 0.47619048, 0.47619048, 0.47619048,
       0.47619048, 0.47619048, 0.47619048, 0.47619048, 0.47619048,
       2.22222222, 2.22222222, 2.22222222, 2.22222222, 2.22222222,
       2.22222222, 2.22222222, 2.22222222, 2.22222222, 2.22222222,
       2.22222222, 2.22222222, 2.22222222, 2.22222222, 2.22222222,
       2.22222222, 2.22222222, 2.22222222, 2.22222222, 2.22222222])

In [ ]:
# Look up the weight for each sampled index
weights_for_samples = w[idx]  # same shape as samples_q
weights_for_samples

array([[0.47619048, 2.22222222, 0.47619048, 0.47619048, 2.22222222],
       [2.22222222, 0.47619048, 0.47619048, 0.47619048, 0.47619048],
       [2.22222222, 0.47619048, 0.47619048, 2.22222222, 0.47619048],
       ...,
       [0.47619048, 0.47619048, 0.47619048, 0.47619048, 0.47619048],
       [0.47619048, 0.47619048, 0.47619048, 2.22222222, 0.47619048],
       [0.47619048, 0.47619048, 0.47619048, 2.22222222, 0.47619048]],
      shape=(10000, 5))

In [73]:
samples_C = weights_for_samples * samples_B
samples_C

array([[23.80952381, 11.11111111, 23.80952381, 23.80952381, 11.11111111],
       [11.11111111, 23.80952381, 23.80952381, 23.80952381, 23.80952381],
       [11.11111111, 23.80952381, 23.80952381, 11.11111111, 23.80952381],
       ...,
       [23.80952381, 23.80952381, 23.80952381, 23.80952381, 23.80952381],
       [23.80952381, 23.80952381, 23.80952381, 11.11111111, 23.80952381],
       [23.80952381, 23.80952381, 23.80952381, 11.11111111, 23.80952381]],
      shape=(10000, 5))

In [74]:
# IS-corrected estimator: average of (weight * value)
estimates_IS = samples_C.mean(axis=1)
estimates_IS

array([18.73015873, 21.26984127, 18.73015873, ..., 23.80952381,
       21.26984127, 21.26984127], shape=(10000,))

In [75]:
print("Mean of estimates :", estimates_IS.mean())  # expect ≈ 20
print("Var  of estimates :", estimates_IS.var())  # expect < 90 ← the prize
print("Bias              :", estimates_IS.mean() - true_mean)

Mean of estimates : 20.003047619047617
Var  of estimates : 6.8382673560090685
Bias              : 0.0030476190476171894


In [ ]:
print(1 / (30 * 0.07))
print(1 / (30 * 0.015))
print((0.476 * 50 + 2.222 * 5) / (2 * (0.476 + 2.222)))

0.47619047619047616
2.2222222222222223
6.469607116382505
